# Unidad 1 · Colab 3 de 3
## Patrones de diseño (Factory, Singleton) y modelado de un negocio digital

**Objetivos de este notebook**

- Entender qué es un patrón de diseño y para qué sirve.
- Implementar el patrón **Factory** para crear objetos sin acoplar el código al constructor específico.
- Implementar el patrón **Singleton** para garantizar una única instancia compartida.
- Aplicar todo lo de la Unidad 1 en el modelado de un negocio digital: `Usuario`, `Carrito`, `Transaccion`, `Suscripcion`.

> **Nivel:** intermedio. Este notebook integra los Colab 1 y 2 de esta unidad.

---

## 1. ¿Qué es un patrón de diseño?

Un **patrón de diseño** es una solución reutilizable a un problema de diseño que aparece una y otra vez en distintos proyectos. No es código que copiás y pegás, sino una forma de organizar las clases para resolver ese problema. Referencia general (catálogo clásico, no oficial de Python pero muy usado como material de estudio): [refactoring.guru](https://refactoring.guru/design-patterns).

## 2. Patrón Factory

**Problema que resuelve:** cuando crear un objeto requiere lógica (decidir qué clase concreta instanciar), esa lógica no debería repetirse en todo el código que necesita crear ese tipo de objeto.

```python
class NotificadorEmail:
    def enviar(self, mensaje):
        print(f'Email: {mensaje}')

class NotificadorSMS:
    def enviar(self, mensaje):
        print(f'SMS: {mensaje}')

def crear_notificador(tipo):
    if tipo == 'email':
        return NotificadorEmail()
    if tipo == 'sms':
        return NotificadorSMS()
    raise ValueError(f'Tipo de notificador desconocido: {tipo}')

notificador = crear_notificador('email')
notificador.enviar('Bienvenido')
```

El código que usa `crear_notificador('email')` no necesita saber que existe la clase `NotificadorEmail` — solo pide un notificador de cierto tipo. Si mañana agregás `NotificadorPush`, solo tocás la fábrica, no cada lugar donde se crean notificadores.

Referencia: [Factory Method](https://refactoring.guru/design-patterns/factory-method)

In [1]:
class NotificadorEmail:
    def enviar(self, mensaje):
        print(f'Email: {mensaje}')

class NotificadorSMS:
    def enviar(self, mensaje):
        print(f'SMS: {mensaje}')

def crear_notificador(tipo):
    if tipo == 'email':
        return NotificadorEmail()
    if tipo == 'sms':
        return NotificadorSMS()
    raise ValueError(f'Tipo de notificador desconocido: {tipo}')

notificador = crear_notificador('email')
notificador.enviar('Bienvenido')

Email: Bienvenido


### Ejercicio 1 — Factory de medios de pago

Creá clases `PagoTarjeta`, `PagoEfectivo` y `PagoTransferencia`, cada una con un método `procesar(self, monto)`, y escribí una función `crear_medio_pago(tipo)` que devuelva la instancia correcta según `tipo` (`'tarjeta'`, `'efectivo'`, `'transferencia'`), lanzando `ValueError` para cualquier otro valor.

<details>
<summary>💡 Ver solución</summary>

```python
class PagoTarjeta:
    def procesar(self, monto):
        return f'Cobrando {monto} con tarjeta'

class PagoEfectivo:
    def procesar(self, monto):
        return f'Cobrando {monto} en efectivo'

class PagoTransferencia:
    def procesar(self, monto):
        return f'Cobrando {monto} por transferencia'

def crear_medio_pago(tipo):
    opciones = {
        'tarjeta': PagoTarjeta,
        'efectivo': PagoEfectivo,
        'transferencia': PagoTransferencia,
    }
    if tipo not in opciones:
        raise ValueError(f'Medio de pago desconocido: {tipo}')
    return opciones[tipo]()

medio = crear_medio_pago('tarjeta')
print(medio.procesar(100))
```

</details>

In [2]:
class PagoTarjeta:
    def procesar(self, monto):
        return f'Cobrando {monto} con tarjeta'

class PagoEfectivo:
    def procesar(self, monto):
        return f'Cobrando {monto} en efectivo'

class PagoTransferencia:
    def procesar(self, monto):
        return f'Cobrando {monto} por transferencia'

def crear_medio_pago(tipo):
    opciones = {
        'tarjeta': PagoTarjeta,
        'efectivo': PagoEfectivo,
        'transferencia': PagoTransferencia,
    }
    if tipo not in opciones:
        raise ValueError(f'Medio de pago desconocido: {tipo}')
    return opciones[tipo]()

medio = crear_medio_pago('tarjeta')
print(medio.procesar(100))

Cobrando 100 con tarjeta


## 3. Patrón Singleton

**Problema que resuelve:** garantizar que una clase tenga una sola instancia en todo el programa (por ejemplo, la configuración de la app o una conexión a base de datos), y que todo el código que la use acceda a esa misma instancia.

```python
class ConfiguracionApp:
    _instancia = None

    def __new__(cls, *args, **kwargs):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.monedas_permitidas = ['ARS', 'USD', 'EUR']
        return cls._instancia

config1 = ConfiguracionApp()
config2 = ConfiguracionApp()

print(config1 is config2)  # True: son el mismo objeto
```

`__new__` controla la creación del objeto (antes de `__init__`); acá lo usamos para devolver siempre la misma instancia guardada en el atributo de clase `_instancia`.

> En Python, un módulo importado ya se comporta como un singleton de forma natural (se ejecuta una sola vez y las importaciones posteriores reutilizan el mismo objeto) — por eso en la práctica es común usar un módulo en vez de esta implementación con `__new__`. Igual vale conocer la versión clásica porque aparece en otros lenguajes y en entrevistas técnicas.

Referencia: [Singleton](https://refactoring.guru/design-patterns/singleton)

In [3]:
class ConfiguracionApp:
    _instancia = None

    def __new__(cls, *args, **kwargs):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.monedas_permitidas = ['ARS', 'USD', 'EUR']
        return cls._instancia

config1 = ConfiguracionApp()
config2 = ConfiguracionApp()

print(config1 is config2)  # True: son el mismo objeto

True


### Ejercicio 2 — Singleton de logging

Implementá una clase `RegistroErrores` (Singleton) con un atributo `errores = []` (lista compartida) y un método `registrar(self, mensaje)` que agregue el mensaje a la lista. Creá dos instancias distintas y confirmá que ambas comparten la misma lista de errores.

<details>
<summary>💡 Ver solución</summary>

```python
class RegistroErrores:
    _instancia = None

    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.errores = []
        return cls._instancia

    def registrar(self, mensaje):
        self.errores.append(mensaje)

r1 = RegistroErrores()
r2 = RegistroErrores()

r1.registrar('Error de conexion')
r2.registrar('Timeout')

print(r1.errores)  # ambos errores, aunque se registraron desde r1 y r2
print(r1 is r2)
```

</details>

In [4]:
class RegistroErrores:
    _instancia = None

    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.errores = []
        return cls._instancia

    def registrar(self, mensaje):
        self.errores.append(mensaje)

r1 = RegistroErrores()
r2 = RegistroErrores()

r1.registrar('Error de conexion')
r2.registrar('Timeout')

print(r1.errores)  # ambos errores, aunque se registraron desde r1 y r2
print(r1 is r2)

['Error de conexion', 'Timeout']
True


## 4. Aplicación práctica: modelado de un negocio digital

Vamos a modelar las entidades centrales de una plataforma digital (por ejemplo, una tienda con suscripciones), integrando todo lo visto en la Unidad 1: clases, encapsulamiento, herencia, polimorfismo, abstracción, métodos especiales, Factory y Singleton.

**Entidades:** `Usuario`, `Producto`, `Carrito`, `Transaccion` (con Factory para el medio de pago), `Suscripcion` (con herencia y polimorfismo entre planes).

In [5]:
class Usuario:
    def __init__(self, nombre, email):
        self.nombre = nombre
        self._email = email

    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, valor):
        if '@' not in valor:
            raise ValueError(f'Email invalido: {valor}')
        self._email = valor

    def __repr__(self):
        return f'Usuario(nombre={self.nombre!r}, email={self._email!r})'


class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

    def __repr__(self):
        return f'Producto(nombre={self.nombre!r}, precio={self.precio})'

ana = Usuario('Ana Perez', 'ana@mail.com')
notebook = Producto('Notebook Lenovo', 599.0)
mouse = Producto('Mouse', 15.0)
print(ana)
print(notebook, mouse)

Usuario(nombre='Ana Perez', email='ana@mail.com')
Producto(nombre='Notebook Lenovo', precio=599.0) Producto(nombre='Mouse', precio=15.0)


## 5. `Carrito`: composición de objetos

Un `Carrito` no hereda de `Producto`, sino que lo contiene (composición: *tiene un(a)* en vez de *es un(a)*).

In [6]:
class Carrito:
    def __init__(self, usuario):
        self.usuario = usuario
        self.items = []

    def agregar(self, producto):
        self.items.append(producto)

    def total(self):
        return sum(p.precio for p in self.items)

    def __len__(self):
        return len(self.items)

    def __repr__(self):
        return f'Carrito(usuario={self.usuario.nombre!r}, items={len(self.items)})'

carrito = Carrito(ana)
carrito.agregar(notebook)
carrito.agregar(mouse)

print(carrito)
print('Items:', len(carrito))
print('Total:', carrito.total())

Carrito(usuario='Ana Perez', items=2)
Items: 2
Total: 614.0


### Ejercicio 3 — `Transaccion` con Factory

Usando `crear_medio_pago` del Ejercicio 1, creá una clase `Transaccion` con `usuario`, `carrito` y `medio_pago` (el string, ej. `'tarjeta'`). En `__init__`, usá la factory para crear el objeto de medio de pago correspondiente y guardalo. Agregá un método `confirmar(self)` que llame a `medio_pago.procesar(self.carrito.total())` y devuelva el resultado.

<details>
<summary>💡 Ver solución</summary>

```python
class Transaccion:
    def __init__(self, usuario, carrito, medio_pago):
        self.usuario = usuario
        self.carrito = carrito
        self.medio_pago = crear_medio_pago(medio_pago)

    def confirmar(self):
        return self.medio_pago.procesar(self.carrito.total())

tx = Transaccion(ana, carrito, 'tarjeta')
print(tx.confirmar())
```

</details>

## 6. `Suscripcion`: herencia y polimorfismo

```python
from abc import ABC, abstractmethod

class Suscripcion(ABC):
    def __init__(self, usuario, precio_base):
        self.usuario = usuario
        self.precio_base = precio_base

    @abstractmethod
    def precio_final(self):
        ...


class SuscripcionMensual(Suscripcion):
    def precio_final(self):
        return self.precio_base


class SuscripcionAnual(Suscripcion):
    DESCUENTO = 0.20

    def precio_final(self):
        return self.precio_base * 12 * (1 - self.DESCUENTO)


suscripciones = [SuscripcionMensual(ana, 10.0), SuscripcionAnual(ana, 10.0)]
for s in suscripciones:
    print(type(s).__name__, '->', round(s.precio_final(), 2))
```

Cada subclase define `precio_final()` a su manera (polimorfismo), y la clase base `Suscripcion` garantiza que toda subclase lo implemente (abstracción).

In [7]:
from abc import ABC, abstractmethod

class Suscripcion(ABC):
    def __init__(self, usuario, precio_base):
        self.usuario = usuario
        self.precio_base = precio_base

    @abstractmethod
    def precio_final(self):
        ...


class SuscripcionMensual(Suscripcion):
    def precio_final(self):
        return self.precio_base


class SuscripcionAnual(Suscripcion):
    DESCUENTO = 0.20

    def precio_final(self):
        return self.precio_base * 12 * (1 - self.DESCUENTO)


suscripciones = [SuscripcionMensual(ana, 10.0), SuscripcionAnual(ana, 10.0)]
for s in suscripciones:
    print(type(s).__name__, '->', round(s.precio_final(), 2))

SuscripcionMensual -> 10.0
SuscripcionAnual -> 96.0


## Mini-proyecto final: sistema completo

Integrá todas las piezas de este notebook en un mini-sistema:

1. Creá 2 `Usuario` distintos.
2. Cada uno arma un `Carrito` con al menos 2 `Producto`.
3. Generá una `Transaccion` para cada carrito, con un medio de pago distinto (usando la Factory).
4. Asigná una `Suscripcion` (mensual o anual) a cada usuario.
5. Usá un `RegistroErrores` (Singleton, del Ejercicio 2) para registrar cualquier error de validación que ocurra (por ejemplo, si intentás crear un `Usuario` con email inválido).
6. Imprimí un resumen final: por cada usuario, su carrito, el resultado de la transacción, y el precio final de su suscripción.

**Entregable:** el código completo del sistema + la salida del resumen final para los 2 usuarios.

---

**Fin de la Unidad 1.** Con estos tres notebooks recorriste el ciclo completo: pasar de scripts lineales a diseño orientado a objetos, aplicar los cuatro pilares de la POO con métodos especiales, y usar patrones de diseño para modelar un negocio digital real — la base sobre la que se construyó el resto del curso (Unidades 2 a 6).

In [8]:
from abc import ABC, abstractmethod
from datetime import datetime
import re

# =====================================================================
# 1. Singleton: RegistroErrores
# =====================================================================
class RegistroErrores:
    _instancia = None

    def __new__(cls):
        if cls._instancia is None:
            cls._instancia = super().__new__(cls)
            cls._instancia.errores = []
        return cls._instancia

    def registrar(self, mensaje: str):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        entrada = f"[{timestamp}] ERROR: {mensaje}"
        self.errores.append(entrada)

    def listar_errores(self):
        return list(self.errores)


# =====================================================================
# 2. Clases Core: Producto, Carrito y Usuario
# =====================================================================
class Producto:
    iva = 0.21

    def __init__(self, nombre: str, precio: float, stock: int):
        self.nombre = nombre
        self.precio = float(precio)
        self.stock = stock

    def precio_con_iva(self) -> float:
        return self.precio * (1 + self.iva)

    def vender(self, cantidad: int) -> int:
        if cantidad <= self.stock:
            vendidas = cantidad
            self.stock -= cantidad
        else:
            vendidas = self.stock
            self.stock = 0
        return vendidas

    def __repr__(self) -> str:
        return f"Producto('{self.nombre}', ${self.precio:.2f})"


class Carrito:
    def __init__(self):
        self.items: list[tuple[Producto, int]] = []

    def agregar(self, producto: Producto, cantidad: int = 1):
        disponibles = producto.vender(cantidad)
        if disponibles > 0:
            self.items.append((producto, disponibles))
        else:
            RegistroErrores().registrar(f"Sin stock para agregar '{producto.nombre}'.")

    def total(self) -> float:
        return sum(prod.precio_con_iva() * cant for prod, cant in self.items)

    def cantidad_items(self) -> int:
        return sum(cant for _, cant in self.items)


class Usuario:
    def __init__(self, nombre: str, email: str):
        self.nombre = nombre
        self.email = email
        self.carrito = Carrito()
        self.suscripcion = None
        self.transaccion = None

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, valor: str):
        patron = r"^[\w\.-]+@[\w\.-]+\.\w+$"
        if not re.match(patron, valor):
            RegistroErrores().registrar(f"Formato de email inválido: '{valor}'.")
            raise ValueError(f"Email no válido: {valor}")
        self._email = valor


# =====================================================================
# 3. Jerarquía y Factory de Pagos (Strategy / Factory Pattern)
# =====================================================================
class MedioPago(ABC):
    @abstractmethod
    def pagar(self, monto: float) -> str:
        pass


class TarjetaCredito(MedioPago):
    def pagar(self, monto: float) -> str:
        return f"Aprobado: ${monto:.2f} con Tarjeta de Crédito (recargo 5% aplicado: ${monto * 1.05:.2f})"


class Transferencia(MedioPago):
    def pagar(self, monto: float) -> str:
        return f"Aprobado: ${monto:.2f} mediante Transferencia Bancaria (sin recargo)"


class MedioPagoFactory:
    @staticmethod
    def crear(tipo: str) -> MedioPago:
        tipo_normalizado = tipo.strip().lower()
        if tipo_normalizado in ("tarjeta", "credito"):
            return TarjetaCredito()
        elif tipo_normalizado in ("transferencia", "banco"):
            return Transferencia()
        else:
            RegistroErrores().registrar(f"Medio de pago no reconocido: '{tipo}'.")
            raise ValueError(f"Tipo de pago desconocido: {tipo}")


class Transaccion:
    def __init__(self, carrito: Carrito, medio_pago: MedioPago):
        self.carrito = carrito
        self.medio_pago = medio_pago
        self.resultado = ""

    def procesar(self) -> str:
        monto_total = self.carrito.total()
        self.resultado = self.medio_pago.pagar(monto_total)
        return self.resultado


# =====================================================================
# 4. Jerarquía de Suscripciones (Polimorfismo)
# =====================================================================
class Suscripcion(ABC):
    def __init__(self, precio_base: float):
        self.precio_base = precio_base

    @abstractmethod
    def precio_final(self) -> float:
        pass

    @abstractmethod
    def tipo(self) -> str:
        pass


class SuscripcionMensual(Suscripcion):
    def tipo(self) -> str:
        return "Mensual"

    def precio_final(self) -> float:
        return self.precio_base


class SuscripcionAnual(Suscripcion):
    def tipo(self) -> str:
        return "Anual (con 20% OFF)"

    def precio_final(self) -> float:
        return self.precio_base * 12 * 0.80


# =====================================================================
# 5. Ejecución del Flujo y Simulación
# =====================================================================
registro = RegistroErrores()

# A. Prueba de error registrado en el Singleton (email inválido)
try:
    usuario_invalido = Usuario("Invalido", "correo_sin_arroba.com")
except ValueError:
    pass

# B. Creación de Productos disponibles
notebook = Producto("Notebook Pro", 800.0, stock=5)
mouse = Producto("Mouse Gamer", 30.0, stock=10)
teclado = Producto("Teclado Mecánico", 75.0, stock=8)
monitor = Producto("Monitor 27\"", 220.0, stock=4)

# C. Creación de 2 Usuarios válidos
usuario1 = Usuario("Ana Ruiz", "ana.ruiz@example.com")
usuario2 = Usuario("Bruno Diaz", "bruno.diaz@example.com")

# D. Carga de carritos (mínimo 2 productos cada uno)
usuario1.carrito.agregar(notebook, 1)
usuario1.carrito.agregar(mouse, 1)

usuario2.carrito.agregar(teclado, 2)
usuario2.carrito.agregar(monitor, 1)

# E. Generación de Transacciones con la Factory (distintos medios de pago)
pago_ana = MedioPagoFactory.crear("tarjeta")
usuario1.transaccion = Transaccion(usuario1.carrito, pago_ana)
usuario1.transaccion.procesar()

pago_bruno = MedioPagoFactory.crear("transferencia")
usuario2.transaccion = Transaccion(usuario2.carrito, pago_bruno)
usuario2.transaccion.procesar()

# F. Asignación de Suscripciones
usuario1.suscripcion = SuscripcionMensual(precio_base=15.0)
usuario2.suscripcion = SuscripcionAnual(precio_base=15.0)


# =====================================================================
# 6. Reporte Final
# =====================================================================
usuarios = [usuario1, usuario2]

print("=" * 70)
print("                      RESUMEN DE OPERACIONES")
print("=" * 70)

for u in usuarios:
    print(f"\n[CLIENTE]: {u.nombre} ({u.email})")
    print("  Items en carrito:")
    for prod, cant in u.carrito.items:
        print(f"    - {prod.nombre} x{cant} (Subtotal c/IVA: ${prod.precio_con_iva() * cant:.2f})")
    print(f"  Total carrito (con IVA): ${u.carrito.total():.2f}")
    print(f"  Resultado de la transacción: {u.transaccion.resultado}")
    print(f"  Plan de Suscripción: {u.suscripcion.tipo()}")
    print(f"  Costo final de la suscripción: ${u.suscripcion.precio_final():.2f}")
    print("-" * 70)

print("\n" + "=" * 70)
print("              REGISTRO CENTRALIZADO DE ERRORES (SINGLETON)")
print("=" * 70)
for error in registro.listar_errores():
    print(f"  {error}")

                      RESUMEN DE OPERACIONES

[CLIENTE]: Ana Ruiz (ana.ruiz@example.com)
  Items en carrito:
    - Notebook Pro x1 (Subtotal c/IVA: $968.00)
    - Mouse Gamer x1 (Subtotal c/IVA: $36.30)
  Total carrito (con IVA): $1004.30
  Resultado de la transacción: Aprobado: $1004.30 con Tarjeta de Crédito (recargo 5% aplicado: $1054.52)
  Plan de Suscripción: Mensual
  Costo final de la suscripción: $15.00
----------------------------------------------------------------------

[CLIENTE]: Bruno Diaz (bruno.diaz@example.com)
  Items en carrito:
    - Teclado Mecánico x2 (Subtotal c/IVA: $181.50)
    - Monitor 27" x1 (Subtotal c/IVA: $266.20)
  Total carrito (con IVA): $447.70
  Resultado de la transacción: Aprobado: $447.70 mediante Transferencia Bancaria (sin recargo)
  Plan de Suscripción: Anual (con 20% OFF)
  Costo final de la suscripción: $144.00
----------------------------------------------------------------------

              REGISTRO CENTRALIZADO DE ERRORES (SINGLETON)
  

======================================================================
                      RESUMEN DE OPERACIONES
======================================================================

[CLIENTE]: Ana Ruiz (ana.ruiz@example.com)
  Items en carrito:
    - Notebook Pro x1 (Subtotal c/IVA: $968.00)
    - Mouse Gamer x1 (Subtotal c/IVA: $36.30)
  Total carrito (con IVA): $1004.30
  Resultado de la transacción: Aprobado: $1004.30 con Tarjeta de Crédito (recargo 5% aplicado: $1054.51)
  Plan de Suscripción: Mensual
  Costo final de la suscripción: $15.00
----------------------------------------------------------------------

[CLIENTE]: Bruno Diaz (bruno.diaz@example.com)
  Items en carrito:
    - Teclado Mecánico x2 (Subtotal c/IVA: $181.50)
    - Monitor 27" x1 (Subtotal c/IVA: $266.20)
  Total carrito (con IVA): $447.70
  Resultado de la transacción: Aprobado: $447.70 mediante Transferencia Bancaria (sin recargo)
  Plan de Suscripción: Anual (con 20% OFF)
  Costo final de la suscripción: $144.00
----------------------------------------------------------------------

======================================================================
              REGISTRO CENTRALIZADO DE ERRORES (SINGLETON)
======================================================================
  [2026-09-08 09:18:07] ERROR: Formato de email inválido: 'correo_sin_arroba.com'.